# 🧠 SVM on Pima Indians Diabetes Dataset
This notebook demonstrates a full machine learning workflow using Support Vector Machines (SVM):
- Data preprocessing (handle missing values, scaling)
- Train/test split
- Baseline model
- Hyperparameter tuning with GridSearchCV
- Visualizations (parameter effects)
- Model evaluation
- Saving and loading model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
%matplotlib inline

In [ ]:
# Load dataset (ensure diabetes.csv is in ../data)
df = pd.read_csv('../data/diabetes.csv')
df.head()

In [ ]:
# Preprocessing
df.columns = [c.strip() for c in df.columns]
target = 'Outcome' if 'Outcome' in df.columns else df.columns[-1]
X = df.drop(columns=[target])
y = df[target]

# Replace zeros with NaN for specific features
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    if col in X.columns:
        X[col] = X[col].replace(0, np.nan)

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_imputed.describe().T

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

In [ ]:
# Baseline SVM model
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', probability=True, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print('Baseline accuracy:', accuracy_score(y_test, y_pred))

In [ ]:
# Hyperparameter tuning
param_grid = {
    'svc__C': [0.01, 0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1]
}
grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1, return_train_score=True)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)

In [ ]:
# Visualize GridSearch results
cv = pd.DataFrame(grid.cv_results_)
pivot = cv.pivot_table(values='mean_test_score', index='param_svc__gamma', columns='param_svc__C')
plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
plt.title('GridSearch mean_test_score (gamma x C)')
plt.show()

In [ ]:
# Underfitting vs Overfitting curve (varying C)
C_values = [0.01, 0.1, 1, 10, 100, 1000]
train_scores, val_scores = [], []
for c in C_values:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel='rbf', C=c, gamma='scale', probability=True, random_state=42))
    ])
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    val_scores.append(np.mean(cross_val_score(model, X_train, y_train, cv=5)))

plt.figure(figsize=(8,6))
plt.plot(C_values, train_scores, marker='o', label='Train Acc')
plt.plot(C_values, val_scores, marker='s', label='Val Acc')
plt.xscale('log')
plt.xlabel('C (log scale)')
plt.ylabel('Accuracy')
plt.title('Effect of C (underfitting vs overfitting)')
plt.legend()
plt.show()

In [ ]:
# Evaluate best model on test set
best = grid.best_estimator_
y_pred_best = best.predict(X_test)
print(classification_report(y_test, y_pred_best))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_best))

In [ ]:
# Save model
joblib.dump({'pipeline': best, 'imputer': imputer}, '../models/svm_pima_pipeline.joblib')
print('Model saved to ../models/svm_pima_pipeline.joblib')